# ДЗ2 NLP: генерация кода, финал v2

По английскому тексту математической задачи генерируем Python `solve()`. Бот проверяет численный результат; отправляем одну программу на вопрос в исходном порядке. Только encoder–decoder Transformer, обучение с нуля на разрешённых исходных парах. Test: **1919 вопросов**, лимит **180 секунд на одной T4**.

### Как запустить с нашими весами

Веса — публичный Kaggle-датасет [thisarivson/nlp-hw02-final-weights](https://www.kaggle.com/datasets/thisarivson/nlp-hw02-final-weights), **версия 4**: три FP16-веса, собственный tokenizer, каталог `code/` с читаемым пакетом и фиксированные входы test/tune. Весь код проекта и история экспериментов — [GitHub](https://github.com/thisariv/hse-nlp-hw2-code-generation).

**Kaggle:** Add Input → датасет `thisarivson/nlp-hw02-final-weights` (версия 4) → Accelerator **GPU T4** → **Run All**. Пути находятся автоматически по `package-manifest.json`.

**Colab:** runtime с T4; перед Run All выполнить в отдельной ячейке (токен Kaggle не нужен):

```python
!pip install -q kagglehub tokenizers sympy numpy
import os, kagglehub
root = kagglehub.dataset_download("thisarivson/nlp-hw02-final-weights/versions/4")
os.environ["HW2_WEIGHTS_ROOT"] = root
os.environ["HW2_CODE_ROOT"] = os.path.join(root, "code")
os.environ["HW2_OUTPUT"] = "/content/hw2-output"
```

Результат — `answers.jsonl` в `OUTPUT_DIR` (на Kaggle: `/kaggle/working/final-package-v2/answers.jsonl`). Установите `tokenizers`, `sympy`, `numpy`, если их нет; измеренный Torch runtime — `2.11.0+cu128` (CUDA 12.8). Загрузка и монтирование датасета не входят в лимит, чтение весов входит.

При двух выделенных T4 используется только GPU0. Вторая карта не участвует. `HW2_DRY_RUN=1` — отдельная CPU-проверка формата на восьми синтетических вопросах, качество она не измеряет.

## Подключение пакета и путей

На Kaggle пути находятся автоматически по `package-manifest.json`. Локально можно задать `HW2_WEIGHTS_ROOT`, `HW2_CODE_ROOT`, `HW2_TEST_PATH`, `HW2_OUTPUT`. Исходники лежат в обычном каталоге `code/hw2_codegen/`; SHA каждого модуля сверяется с manifest перед импортом.

In [1]:
import os
import json
import hashlib
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
DRY_RUN = os.environ.get("HW2_DRY_RUN", "0") == "1"
OUTPUT_DIR = Path(os.environ.get("HW2_OUTPUT", "/kaggle/working/final-package-v2" if Path("/kaggle").exists() else "output/final-package-v2"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if DRY_RUN:
    WEIGHTS_ROOT = Path(".")
else:
    root_override = os.environ.get("HW2_WEIGHTS_ROOT")
    manifests = [Path(root_override) / "package-manifest.json"] if root_override else list(Path("/kaggle/input").rglob("package-manifest.json"))
    assert len(manifests) == 1, "Добавьте dataset nlp-hw02-final-weights или задайте HW2_WEIGHTS_ROOT"
    WEIGHTS_ROOT = manifests[0].parent
    manifest = json.loads(manifests[0].read_text())

CODE_ROOT = Path(os.environ.get("HW2_CODE_ROOT", str(WEIGHTS_ROOT / "code")))
assert (CODE_ROOT / "hw2_codegen").is_dir(), "Нужен каталог code/hw2_codegen"
if not DRY_RUN:
    for name, spec in manifest["code_files"].items():
        path = CODE_ROOT / Path(name).relative_to("code")
        assert path.stat().st_size == spec["bytes"] and hashlib.sha256(path.read_bytes()).hexdigest() == spec["sha256"], name
TEST_PATH = Path(os.environ.get("HW2_TEST_PATH", str(WEIGHTS_ROOT / "test.json")))
print("Code:", CODE_ROOT, "| Weights:", WEIGHTS_ROOT, "| Test:", TEST_PATH)


Code: /kaggle/input/datasets/thisarivson/nlp-hw02-final-weights/code | Weights: /kaggle/input/datasets/thisarivson/nlp-hw02-final-weights | Test: /kaggle/input/datasets/thisarivson/nlp-hw02-final-weights/test.json


## Представление и модели

Собственный BPE: 8189 базовых токенов + PAD/BOS/EOS + три compact-mode токена = **8195**. Цифры кодируются отдельно. Compact удаляет комментарии и сжимает стандартные header/tail `solve`; восстановление Python выполняется до исполнения. Новые обучающие ответы не создавались.

| Модель (порядок ансамбля) | Encoder + decoder | Ширина / головы | Особенность |
|---|---:|---:|---|
| W B7 — greedy-якорь | 6 + 6 | 768 / 12 | Headwise Q/K LayerNorm; z-loss при обучении; B7 cooldown |
| C-long2x B7 | 6 + 6 | 512 / 8 | Длинные train-программы предъявлялись дважды; B7 cooldown |
| P1-B | 6 + 6 | 512 / 8 | Ветка C с length-stratified loss truncation q=0.1 при cooldown |

Общие элементы: Pre-LN, SwiGLU, learned positions, отдельная выходная голова. Q/K нормируются до SDPA и в KV-cache. Z-loss — регуляризация обучения. Все веса model-only FP16, загрузка `weights_only=True`, строгая проверка state dict и SHA. B7 — output Kaggle-запуска `nlp-hw02-b7` v1 (аккаунт друга), points `model-GPU0/points/b6-final` (W) и `model-GPU1/points/b6-final` (C); P1-B — прежний output v11.


## Ансамбль и бюджет

1. Greedy W B7 и по одному stochastic sample каждой модели (T=0.7, seed=270042).
2. Если четыре программы возвращают одинаковое конечное число (`isclose`, 1e-6), остановиться.
3. Иначе добавить по семь samples C-long2x B7 и P1-B: **1/1/1 → 1/8/8**, максимум 18 программ вместе с greedy.
4. Голосование по компонентам численных ответов: число голосов → наличие greedy → средний log p → стабильный порядок. Ошибки исполнения не голосуют. Выбранный код независимо переисполняется.

Fast decoder v11 использует предвыделенный self-KV, cross-KV sharing при K=1, compaction завершённых строк, async EOS и Exp(1) sampling. CPU-исполнение перекрывается с GPU; одинаковый код исполняется один раз, сохраняя все голоса.

**Sequence budget 640** ограничивает число одновременно декодируемых последовательностей; batch вопросов = budget // K. Allocator `expandable_segments:True,garbage_collection_threshold:0.8` задаётся до импорта Torch, свободный CUDA cache освобождается между batches. Код pipeline, seed, batching и политика совпадают с вариантом v2 из пяти финальных вариантов.

Политика и параметры находятся в одном `hw2_codegen/config.json`. Ранее вариант v2: public **80,10%** (сообщение пользователя), tune **1557/2048**, cold test **111,421 с**. Ниже сохраняется отдельная проверка финальной упаковки. Guards: генерация до 160 с, исполнение до 166, проверка до 170, supervisor до 172. Успех подтверждается полным таймером.


In [2]:
import sys
sys.path.insert(0, str(CODE_ROOT.resolve()))
from hw2_codegen.config import default_config
from hw2_codegen.runner import run_cold

config = default_config()
config.update(weights_root=str(WEIGHTS_ROOT.resolve()), tokenizer=str((WEIGHTS_ROOT / "tokenizer.json").resolve()),
              rows=str(TEST_PATH.resolve()), output=str((OUTPUT_DIR / "test-1").resolve()),
              device="cpu" if DRY_RUN else "cuda:0", dry_run=DRY_RUN)
print("Budget:", config["sequence_budget"], "| Models:", [(m["name"], m["K1"], m["K2"]) for m in config["models"]])


Budget: 640 | Models: [('W-B7', 1, 1), ('C-long2x-B7', 1, 8), ('pilot-b', 1, 8)]


## Один cold test → answers.jsonl

Таймер начинается **до создания свежего Python-процесса**. Он включает импорт Torch/пакета, чтение test/tokenizer/весов, SHA, загрузку, warmup, GPU decode, восстановление Python, CPU execution/vote, независимое переисполнение winners и закрытие `answers.jsonl`. До конца greedy отсутствие fallback считается ошибкой. При guard ответы помечаются guarded; такой прогон не проходит финальную проверку.

Повторный запуск требует нового `OUTPUT_DIR`: готовый результат не перезаписывается. Следующая ячейка запускает test ровно один раз.

In [3]:
test_result = run_cold(config)
answers_path = OUTPUT_DIR / "answers.jsonl"
answers_path.write_bytes((OUTPUT_DIR / "test-1/answers.jsonl").read_bytes())
answers = [json.loads(line) for line in answers_path.read_text().splitlines()]
assert len(answers) == (8 if DRY_RUN else 1919)
assert all(set(row) == {"code"} and isinstance(row["code"], str) for row in answers)
print(json.dumps({k: test_result.get(k) for k in ["status", "rows", "elapsed_seconds", "peak_allocated_bytes", "peak_reserved_bytes", "peak_device_used_bytes", "guards", "oom_attempts"]}, indent=2))
print("Bot output:", answers_path)


{
  "status": "complete",
  "rows": 1919,
  "elapsed_seconds": 122.65776955000001,
  "peak_allocated_bytes": 11008209408,
  "peak_reserved_bytes": 11123294208,
  "peak_device_used_bytes": 11759976448,
  "guards": [],
  "oom_attempts": []
}
Bot output: /kaggle/working/final-package-v2/answers.jsonl


## Контроль tune2048

Это отдельный свежий процесс после test, его время не прибавляется к лимиту test. Hash-split фиксирован заранее; tune многократно использовался в исследованиях и не является независимым test. Reference передаётся только диагностике после записи ответов, выбор программ не видит gold. Политика не меняется по результату.

Для обычного проверяющего достаточно `RUN_TUNE=False`; в проверке финальной упаковки оставлено True, так что **Run All** выполняет test1919 + tune2048.

In [4]:
RUN_TUNE = True
summary = {"test": test_result, "training": False, "policy": "v2-budget640"}
if RUN_TUNE and not DRY_RUN:
    tune_config = dict(config, rows=str((WEIGHTS_ROOT / "tune2048.json").resolve()),
                       reference=str((WEIGHTS_ROOT / "tune2048-reference.json").resolve()),
                       output=str((OUTPUT_DIR / "tune").resolve()))
    tune_result = run_cold(tune_config)
    quality = json.loads((OUTPUT_DIR / "tune/quality.json").read_text())
    summary.update(tune=tune_result, quality=quality)
    print("Tune:", quality["matches"], "/", quality["n"], "| time:", round(tune_result["elapsed_seconds"], 3))
    print("Tune memory:", {k: tune_result.get(k) for k in ["peak_allocated_bytes", "peak_reserved_bytes", "peak_device_used_bytes"]})
runs = [summary[k] for k in ["test", "tune"] if k in summary]
summary["gates"] = dict(test_time=test_result["elapsed_seconds"] <= 150,
    complete=all(r["status"] == "complete" and not r.get("fallback_rows") and not r.get("partial_rows") for r in runs),
    reexecution=all(not r.get("verification", {}).get("mismatch_rows", []) for r in runs),
    no_oom=all(not r.get("oom_attempts") for r in runs))
if not DRY_RUN:
    summary["gates"]["memory"] = all(isinstance(r.get("peak_reserved_bytes"), int) and r["peak_reserved_bytes"] <= 14_000_000_000 for r in runs)
    if "quality" in summary:
        summary["gates"]["tune"] = abs(summary["quality"]["matches"] - config["expected_tune_matches"]) <= .01 * summary["quality"]["n"]
summary["status"] = "passed" if all(summary["gates"].values()) else "review_required"
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n")
print("Gates:", summary["gates"], "| status:", summary["status"])
assert test_result["within_budget"], "Превышен официальный лимит 180 с"


Tune: 1560 / 2048 | time: 107.987
Tune memory: {'peak_allocated_bytes': 8412472320, 'peak_reserved_bytes': 8487174144, 'peak_device_used_bytes': 9123856384}
Gates: {'test_time': True, 'complete': True, 'reexecution': True, 'no_oom': True, 'memory': True, 'tune': True} | status: passed


Скачать `answers.jsonl` — одна строка `{"code": "..."}` на вопрос. Для аудита сохранить также `summary.json`, `test-1/timing.json`, исполненный notebook и каталог `tune/` (answers, quality, journal). Датасет весов публичный; Kaggle-ноутбук финального прогона приватный, его outputs сохранены ниже.

CPU parity проверяет одинаковые budget/precision на нескольких строках относительно исполняемого frozen v11. GPU quality/time/memory проверяются отдельным cold запуском. Полное историческое обучение не перезапускается; код и журналы — в каталоге `work/` [репозитория](https://github.com/thisariv/hse-nlp-hw2-code-generation).

## Сохранённый финальный прогон v2 — 21.10.2026

Один private push на `thisarivson/nlp-hw02-final-package-v2-budget640`, v1,
dataset `thisarivson/nlp-hw02-final-weights`, v4. Серверный код проверен;
ниже в code cells сохранены реальные outputs исполненного notebook.

| Split | Cold, с | Torch allocated / reserved, GB | NVML peak used, GB |
|---|---:|---:|---:|
| Test1919 | **122,658** | 11,008 / **11,123** | **11,760** |
| Tune2048 | **107,987** | 8,412 / **8,487** | **9,124** |

GB=10⁹bytes. Tune **1560/2048 = 76,171875%**. Все gates passed:
OOM/guards/partial/fallback/non-numeric/mismatch —0; tune journal replay2048 exact.
Ответы: `/kaggle/working/final-package-v2/answers.jsonl`;
локальная копия — `work/submissions/final-package-v2.jsonl`.

С прежней посылкой v2 нет побайтного совпадения: **279 программ /159 численных
ответов** отличаются; обе посылки независимо исполнены. Config/веса/seed/budget
и исполняемый код те же. Различия начинаются в stage1 samples C/P1-B;
вероятный механизм — timing async EOS/compaction и связанный расход RNG.
Точные поэтапные CUDA-трассы отсутствуют. Public **80,10%** относится к прежней
v2.jsonl, эта посылка дала на public **79,37%** — разброс между прогонами (см. README). Политика не менялась.
